# TP2c — Validation et surapprentissage

Au TP2b, le coût baissait toujours quand on agrandissait le réseau ou qu'on l'entraînait plus longtemps.
Mais ce coût était mesuré sur les appartements que le modèle avait appris.

Ce qui compte, c'est que le modèle **généralise** : qu'il prédise bien le prix d'appartements qu'il n'a **jamais vus**.
Pour le mesurer, on met de côté une partie des exemples : le **dataset de validation** (*validation dataset*).

# Les données et le modèle

Les mêmes qu'au TP2b. Exécutez les cellules.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

from carte_prix import trace_carte_prix

df = pd.read_csv('data/bordeaux_appartements_2025.csv.gz')

cols = {'valeur_fonciere': 'prix', 'surface_reelle_bati': 'surface', 'nombre_pieces_principales': 'nb_pieces'}
examples = df.rename(columns=cols, errors='raise')[['prix', 'surface', 'nb_pieces', 'latitude', 'longitude']]

features = ['surface', 'nb_pieces', 'latitude', 'longitude']
X = torch.tensor(examples[features].values, dtype=torch.float32)
Y = torch.tensor(examples[['prix']].values, dtype=torch.float32)

In [ ]:
class NeuralNetworkModel(nn.Module):
    def __init__(self, X, Y, M, activation=nn.ReLU()):
        super().__init__()
        # une moyenne et un écart-type par colonne de X, puis pour les prix
        self.X_mean = X.mean(dim=0)
        self.X_std = X.std(dim=0)
        self.Y_mean = Y.mean()
        self.Y_std = Y.std()

        self.hidden = nn.Linear(4, M)
        self.activation = activation
        self.output = nn.Linear(M, 1)

    def forward(self, x):
        x = (x - self.X_mean) / self.X_std      # on standardise les caractéristiques
        z = self.hidden(x)                      # couche cachée
        h = self.activation(z)                  # fonction d'activation
        y = self.output(h)                      # couche de sortie : un prix standardisé
        return y * self.Y_std + self.Y_mean     # on revient aux euros

# 1. Séparer entraînement et validation

On mélange les appartements au hasard, puis on garde 80 % pour l'entraînement et 20 % pour la validation.

In [ ]:
torch.manual_seed(0)
ids = torch.randperm(len(X))         # les numéros de ligne 0 à N-1, dans le désordre
n_train = int(0.8 * len(X))
train_ids = ids[:n_train]            # les 80 % premiers
val_ids = ids[n_train:]              # les 20 % restants

X_train, Y_train = X[train_ids], Y[train_ids]
# TODO le dataset de validation
X_val, Y_val = ...

print('entraînement :', X_train.shape)
print('validation   :', X_val.shape)

Le modèle calcule la moyenne et l'écart-type des données qu'on lui donne à sa création.
On lui donnera **uniquement** `X_train` et `Y_train` : le dataset de validation ne doit servir à rien pendant l'entraînement,
pas même à la standardisation.

# 2. Entraîner en surveillant la validation

`train_with_validation()` est la fonction `train()` du TP2b, avec une mesure supplémentaire à chaque epoch :
la RMSE sur le dataset de validation. Cette mesure ne sert qu'à observer : elle ne modifie pas les paramètres.

In [ ]:
def train_with_validation(model, X_train, Y_train, X_val, Y_val, epochs, alpha):
    cost_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=alpha)

    rmse_train, rmse_val = [], []
    for epoch in range(epochs):
        # entraînement : les 4 étapes, sur le dataset d'entraînement seulement
        cost = cost_function(model(X_train), Y_train)
        cost.backward()
        optimizer.step()
        optimizer.zero_grad()
        rmse_train.append(cost.item() ** 0.5)

        # validation : une simple inférence sur le dataset de validation
        with torch.no_grad():
            cost_val = cost_function(model(X_val), Y_val)
        rmse_val.append(cost_val.item() ** 0.5)

    best_epoch = rmse_val.index(min(rmse_val))
    print(f'meilleure epoch : {best_epoch}, RMSE validation : {rmse_val[best_epoch]:,.0f} €')
    print(f'dernière epoch  : {epochs - 1}, RMSE validation : {rmse_val[-1]:,.0f} €')
    return rmse_train, rmse_val, best_epoch


def plot_learning_curves(rmse_train, rmse_val, best_epoch, skip=100):
    plt.plot(range(skip, len(rmse_train)), rmse_train[skip:], label='entraînement')
    plt.plot(range(skip, len(rmse_val)), rmse_val[skip:], label='validation')
    plt.axvline(best_epoch, color='grey', linestyle='--', label=f'meilleure epoch ({best_epoch})')
    plt.xlabel('epoch')
    plt.ylabel('RMSE [€]')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
torch.manual_seed(0)
model = NeuralNetworkModel(X_train, Y_train, M=200)
rmse_train, rmse_val, best_epoch = train_with_validation(model, X_train, Y_train, X_val, Y_val,
                                                         epochs=5000, alpha=0.001)
plot_learning_curves(rmse_train, rmse_val, best_epoch)

## Questions

1. À quelle epoch le coût de validation est-il le plus bas ?
2. Après cette epoch, que fait le coût d'entraînement ? Et le coût de validation ? Comment s'appelle ce phénomène ?
3. Au TP2b (question 7), le coût baissait toujours quand on entraînait plus longtemps. Est-ce que cela voulait dire
   que le modèle s'améliorait ?

## Réponses

TODO


# 3. L'arrêt anticipé

La solution la plus simple : arrêter l'entraînement quand le coût de validation ne s'améliore plus.
C'est l'**arrêt anticipé** (*early stopping*).

Pendant l'entraînement, on ne sait pas encore quelle sera la meilleure epoch. `train_with_early_stopping()` procède donc ainsi :

- à chaque nouveau meilleur coût de validation, elle retient une copie des paramètres du modèle ;
- si `patience` epochs passent sans amélioration, elle s'arrête ;
- à la fin, elle remet les paramètres retenus : le modèle est celui de la meilleure epoch.

Complétez la condition d'arrêt.

In [ ]:
import copy


def train_with_early_stopping(model, X_train, Y_train, X_val, Y_val, epochs, alpha, patience):
    cost_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=alpha)

    best_rmse, best_epoch, best_params = float('inf'), 0, None
    for epoch in range(epochs):
        cost = cost_function(model(X_train), Y_train)
        cost.backward()
        optimizer.step()
        optimizer.zero_grad()

        with torch.no_grad():
            rmse_val = cost_function(model(X_val), Y_val).item() ** 0.5

        # nouveau meilleur modèle : on retient une copie de ses paramètres
        if rmse_val < best_rmse:
            best_rmse, best_epoch = rmse_val, epoch
            best_params = copy.deepcopy(model.state_dict())

        # TODO si `patience` epochs sont passées depuis best_epoch, sortir de la boucle avec break

    model.load_state_dict(best_params)   # on revient au meilleur modèle
    print(f'meilleure epoch : {best_epoch}, RMSE validation : {best_rmse:,.0f} €')

In [ ]:
torch.manual_seed(0)
best_model = NeuralNetworkModel(X_train, Y_train, M=200)
train_with_early_stopping(best_model, X_train, Y_train, X_val, Y_val, epochs=5000, alpha=0.001, patience=200)

## Questions

4. À quelle epoch l'entraînement s'arrête-t-il ? Combien d'epochs a-t-on économisées par rapport aux 5 000 ?
5. Regardez la courbe de validation de la partie 2 : pourquoi ne pas s'arrêter dès que le coût de validation remonte
   une seule fois ?

## Réponses

TODO


# 4. Voir le surapprentissage

Comparons les cartes de prix du modèle arrêté à la meilleure epoch (`best_model`) et du modèle entraîné
jusqu'au bout (`model`).

In [ ]:
trace_carte_prix(best_model, examples, surface=50, nb_pieces=3, rmse_val=rmse_val[best_epoch],
                 titre=f'Meilleur modèle : arrêt anticipé (epoch {best_epoch})')
plt.show()
trace_carte_prix(model, examples, surface=50, nb_pieces=3, rmse_val=rmse_val[-1],
                 titre=f'Modèle surentraîné : {len(rmse_val)} epochs')
plt.show()

## Questions

6. Comparez les deux cartes : la forme des zones, l'échelle des couleurs, l'emplacement du quartier le plus cher.
   Laquelle vous semble la plus réaliste ?
7. Pour choisir les hyperparamètres (`M`, `alpha`, nombre d'epochs…), quel coût faut-il regarder ? Pourquoi ?

## Réponses

TODO
